# Aphasia vs. Control Classification with DeBERTa‑v3‑Base

This notebook fine‑tunes a DeBERTa‑v3‑base model on a small, mixed dataset of 100 `.cha` transcripts (50 aphasia, 50 control) and optionally benchmarks it against GPT- 4 zero‑shot classification.


## 1. Install Dependencies

We need Transformers, Datasets, PyTorch, scikit‑learn, and OpenAI for the optional baseline.


In [ ]:
!pip install --quiet transformers datasets torch scikit-learn openai


## 2. Load & Shuffle `.cha` Transcripts

Read all `.cha` files from `aphasia/` (label 1) and `control/` (label 0), shuffle them together.


In [ ]:
# Load transcripts and assign labels

import os, glob, random, re
from sklearn.model_selection import train_test_split
import torch

DATA_ROOT = '/content/'        # Adjust if using Drive: '/content/drive/MyDrive/cha_data'
aphasia_dir = os.path.join(DATA_ROOT, 'aphasia')
control_dir = os.path.join(DATA_ROOT, 'control')

# Collect file paths
aphasia_paths = glob.glob(os.path.join(aphasia_dir, '*.cha'))
control_paths = glob.glob(os.path.join(control_dir, '*.cha'))

texts, labels = [], []

# Read aphasia transcripts (label=1)
for p in aphasia_paths:
    with open(p, 'r', encoding='utf-8', errors='ignore') as f:
        texts.append(f.read())
        labels.append(1)

# Read control transcripts (label=0)
for p in control_paths:
    with open(p, 'r', encoding='utf-8', errors='ignore') as f:
        texts.append(f.read())
        labels.append(0)

# Shuffle the combined dataset
combined = list(zip(texts, labels))
random.seed(42)
random.shuffle(combined)
texts, labels = zip(*combined)
texts, labels = list(texts), list(labels)

# Quick sanity check
print(f"Total transcripts: {len(texts)}")
print(f"  Aphasia: {sum(labels)}")
print(f"  Control: {len(labels)-sum(labels)}")


## 3. Inspect & Split Dataset

Peek at one example per class, then do an 80/20 stratified train/test split.


In [ ]:
# Inspect few transcripts and split
from sklearn.model_selection import train_test_split

# Peek at example transcripts
print("→ Aphasia sample (first 300 chars):\n", texts[labels.index(1)][:300], "\n")
print("→ Control sample (first 300 chars):\n", texts[labels.index(0)][:300], "\n")

# Stratified split into train/test
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

print(f"Training samples: {len(train_texts)}")
print(f"Testing samples:  {len(test_texts)}")

## 4. Load DeBERTa‑v3‑Base Tokenizer & Model

We’ll fine‑tune this model for binary classification (2 labels).


In [ ]:
# Load tokenizer and pre-trained model
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

## 5. Tokenize the Texts

Convert raw text into input IDs + attention masks for DeBERTa.


In [ ]:
# Tokenization
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=512
)
test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=512
)

## 6. Build a PyTorch Dataset

Wrap encodings and labels into a Dataset class for the Trainer API.


In [ ]:
# Create Dataset class for Trainer
import torch

class AphasiaDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx])
                for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = AphasiaDataset(train_encodings, train_labels)
test_dataset  = AphasiaDataset(test_encodings,  test_labels)


## 7. Define Evaluation Metrics

Compute accuracy, precision, recall, and F1 on the binary task.


In [ ]:
# Metrics function
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


## 8. Configure Trainer & Fine‑Tune

Set up `TrainingArguments` and train the model.


In [ ]:
# Cell X: Memory‑Safe Trainer Setup (fixed checkpointing)

from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch
from transformers import (
    Trainer, TrainingArguments, EarlyStoppingCallback
)

# Custom loss with class weights
def compute_loss(model, inputs, return_outputs=False):
    labels = inputs.pop("labels").to(model.device) # Move labels to model's device
    outputs = model(**inputs)
    logits = outputs.logits
    loss_fn = torch.nn.CrossEntropyLoss()
    loss = loss_fn(logits, labels)
    return (loss, outputs) if return_outputs else loss



In [ ]:
# TrainingArguments with gradient_checkpointing flag
training_args = TrainingArguments(
    output_dir="./deberta_results",
    seed=42,
    num_train_epochs=8,
    per_device_train_batch_size=4,        # lower to reduce memory
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,        # effective batch size = 16
    learning_rate=2e-5,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True,          # ← turn on checkpointing here
    report_to=[]
)

# Initialize Trainer (no manual training_step override)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)



In [ ]:
# Cell 7: Train & Validation
print("→ Training…")
trainer.train()
# Override the compute_loss method of the Trainer instance
trainer.compute_loss = compute_loss
print("\n→ Best found model metrics on validation:")
print(trainer.evaluate())


In [ ]:
# Cell 8: Final Test Evaluation
print("→ Evaluating on held‑out test set:")
test_out = trainer.predict(test_dataset)
metrics = compute_metrics((test_out.predictions, test_out.label_ids))
print(metrics)

## 9. Save the Fine‑Tuned Model

After training, save both model and tokenizer so you can reload later without retraining.


In [ ]:
SAVE_DIR="./deberta_saved"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"model and tokentizer saved to {SAVE_DIR}")

## 10. Reload Saved Model for Inference

Load the saved model/tokenizer and run inference on the test set without retraining.


In [ ]:
# Load and evaluate saved model

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch.nn.functional as F

# Load from disk
SAVE_DIR = "./deberta_saved"
loaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
loaded_model     = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

## 11. Evaluation function for the Fine‑Tuned Model

After training the model, function to display the evaluation metrics.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch.nn.functional as F

def display_metrics(loaded_model, loaded_tokenizer, test_texts, test_labels):
    """
    Calculate and display evaluation metrics using loaded model and tokenizer.
    """

    # Tokenize test set
    encodings = loaded_tokenizer(
        test_texts,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    # Inference
    with torch.no_grad():
        outputs = loaded_model(**encodings)
        probabilities = F.softmax(outputs.logits, dim=-1)
        threshold = 0.4  # Experiment with lower values like 0.4, 0.3, etc.
        preds = (probabilities[:, 1] > threshold).cpu().numpy().astype(int)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average="binary")
    acc = accuracy_score(test_labels, preds)

    # Display metrics
    print("Evaluation Metrics:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1: {f1:.4f}")

## 12. Plot curves and Training Loss

plot_training_curves(trainer) for plotting loss & F1 per epoch.

In [ ]:
#Function to plot training and evaluation curves
import matplotlib.pyplot as plt

def plot_training_curves(trainer):
    """
    Plot training loss, evaluation loss, and evaluation F1 score per epoch from the Trainer's log history.
    """
    log_history = trainer.state.log_history

    # Extract training loss entries per epoch (average loss logged at end of each epoch)
    train_loss_epochs = [entry['epoch'] for entry in log_history if 'loss' in entry and 'eval_loss' not in entry]
    train_losses      = [entry['loss']  for entry in log_history if 'loss' in entry and 'eval_loss' not in entry]

    # Extract evaluation loss entries per epoch
    eval_loss_epochs = [entry['epoch'] for entry in log_history if 'eval_loss' in entry]
    eval_losses      = [entry['eval_loss']  for entry in log_history if 'eval_loss' in entry]

    # Extract evaluation F1 entries per epoch
    eval_f1_epochs = [entry['epoch'] for entry in log_history if 'eval_f1' in entry]
    eval_f1_scores = [entry['eval_f1'] for entry in log_history if 'eval_f1' in entry]

    # Plot Training Loss
    plt.figure()
    plt.plot(train_loss_epochs, train_losses, marker='o', label='Train Loss')
    if eval_loss_epochs:
        plt.plot(eval_loss_epochs, eval_losses, marker='x', label='Eval Loss')
    plt.title('Training and Evaluation Loss per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Plot Evaluation F1 Score (if multiple points exist)
    if len(eval_f1_scores) > 1:
        plt.figure()
        plt.plot(eval_f1_epochs, eval_f1_scores, marker='o')
        plt.title('Evaluation F1 Score per Epoch')
        plt.xlabel('Epoch')
        plt.ylabel('F1 Score')
        plt.grid(True)
        plt.show()
    else:
        print(f"Only one evaluation point (epoch {eval_f1_epochs[0]}), skipping F1 plot.")


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

def plot_precision_recall_curve(model, tokenizer, texts, labels, highlight_threshold=None):
    """
    Tokenizes the texts, runs the model to get predicted probabilities for the positive class,
    computes the precision–recall curve, and plots it. Optionally highlights a specific threshold.
    """
    # Tokenize the test set
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    model.eval()
    with torch.no_grad():
        outputs = model(**encodings)
        probs = F.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()

    # Compute Precision-Recall curve and average precision
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    avg_prec = average_precision_score(labels, probs)

    # Plot
    plt.figure(figsize=(6, 6))
    plt.plot(recall, precision, marker='.', label=f'AP = {avg_prec:.2f}')
    plt.title('Precision–Recall Curve')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.legend(loc='lower left')
    plt.grid(True)

    # Optionally highlight a specific decision threshold
    if highlight_threshold is not None:
        # Find the index of the closest threshold
        idx = np.argmin(np.abs(thresholds - highlight_threshold))
        plt.scatter(recall[idx], precision[idx], color='red', s=100,
                    label=f'Threshold = {highlight_threshold:.2f}')
        plt.legend(loc='lower left')

    plt.show()





Invocation snippet to evaluate, display metrics, and plot curves with your trainer.

In [ ]:
# Invocation to display metrics and plot curves
# After training or loading the model, run:
metrics = trainer.evaluate()

# Display numeric results
display_metrics(loaded_model, loaded_tokenizer, test_texts, test_labels)

# Plot training and evaluation curves
plot_training_curves(trainer)

In [ ]:
plot_precision_recall_curve(loaded_model, loaded_tokenizer, test_texts, test_labels, highlight_threshold=0.4)

In [ ]:
!zip -r deberta_saved.zip deberta_saved
!zip -r deberta_results.zip deberta_results